## Pytorch minimal workshop

### How to load data
First, import crucial dependencies

In [43]:

import numpy as np
import pandas as pd

import torch

# If you want to run PyTorch on GPU, you need to run a different torch setup 
# (now defined as additional dependency, to be activated by uv sync --extra cu126 
# This is highly dependent on your architecture!! Let me know if that is something of interest!

# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))




Load a dataset using pandas. Pandas is still quite convenient for working with data, but ultimately, pytorch needs tensors (which are, just like DataFrames, built on top of numpy arrays). Any datasplits should also be done before transforming the dataframes into tensors.

In [44]:
from sklearn.datasets import load_diabetes
import pandas as pd

data = load_diabetes()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.DataFrame(data.target)

print(y.shape)
print(X.head())

(442, 1)
        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  
0 -0.002592  0.019907 -0.017646  
1 -0.039493 -0.068332 -0.092204  
2 -0.002592  0.002861 -0.025930  
3  0.034309  0.022688 -0.009362  
4 -0.002592 -0.031988 -0.046641  


There are different ways to convert a dataframe into a tensor:

1) Directly from dataframe, via the `values` property.

In [45]:
X_tensor = torch.tensor(X.values)
X_tensor

tensor([[ 0.0381,  0.0507,  0.0617,  ..., -0.0026,  0.0199, -0.0176],
        [-0.0019, -0.0446, -0.0515,  ..., -0.0395, -0.0683, -0.0922],
        [ 0.0853,  0.0507,  0.0445,  ..., -0.0026,  0.0029, -0.0259],
        ...,
        [ 0.0417,  0.0507, -0.0159,  ..., -0.0111, -0.0469,  0.0155],
        [-0.0455, -0.0446,  0.0391,  ...,  0.0266,  0.0445, -0.0259],
        [-0.0455, -0.0446, -0.0730,  ..., -0.0395, -0.0042,  0.0031]])

2. Via `torch.tensor` but via numpy array. Likewise, numpy arrays can be directly converted into tensors.

In [46]:
X_tensor = torch.tensor(X.to_numpy(), dtype=torch.float32) # datatype conversion optional
X_tensor

tensor([[ 0.0381,  0.0507,  0.0617,  ..., -0.0026,  0.0199, -0.0176],
        [-0.0019, -0.0446, -0.0515,  ..., -0.0395, -0.0683, -0.0922],
        [ 0.0853,  0.0507,  0.0445,  ..., -0.0026,  0.0029, -0.0259],
        ...,
        [ 0.0417,  0.0507, -0.0159,  ..., -0.0111, -0.0469,  0.0155],
        [-0.0455, -0.0446,  0.0391,  ...,  0.0266,  0.0445, -0.0259],
        [-0.0455, -0.0446, -0.0730,  ..., -0.0395, -0.0042,  0.0031]])

3. Using `DataLoader` for large datasets (data larger than allocated memory)

In [47]:
from torch.utils.data import DataLoader, TensorDataset

X_tensor = torch.tensor(X.to_numpy())
y_tensor = torch.tensor(y.to_numpy())

dataset = TensorDataset(X_tensor, y_tensor)

dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

for batch in dataloader:
    print(batch)

[tensor([[ 0.0163,  0.0507, -0.0461,  0.0115, -0.0332, -0.0160, -0.0103, -0.0026,
         -0.0440, -0.0425],
        [ 0.0090, -0.0446, -0.0224, -0.0321, -0.0497, -0.0686,  0.0781, -0.0709,
         -0.0629, -0.0384]]), tensor([[47.],
        [84.]])]
[tensor([[ 0.0272,  0.0507,  0.0940,  0.0976, -0.0346, -0.0320, -0.0434, -0.0026,
          0.0366,  0.1066],
        [-0.0709,  0.0507, -0.0892, -0.0745, -0.0428, -0.0257, -0.0324, -0.0026,
         -0.0129, -0.0549]]), tensor([[263.],
        [104.]])]
[tensor([[ 0.0163, -0.0446,  0.0736, -0.0412, -0.0043, -0.0135, -0.0139, -0.0011,
          0.0429,  0.0445],
        [ 0.0163, -0.0446, -0.0288, -0.0091, -0.0043, -0.0098,  0.0450, -0.0395,
         -0.0307, -0.0425]]), tensor([[275.],
        [179.]])]
[tensor([[ 0.0018, -0.0446, -0.0396, -0.1009, -0.0291, -0.0301,  0.0450, -0.0502,
         -0.0683, -0.1295],
        [ 0.0090, -0.0446,  0.0552, -0.0057,  0.0576,  0.0447, -0.0029,  0.0232,
          0.0557,  0.1066]]), tensor([[ 65.],


4. Use a Custom Dataset Class for more flexibility and cleaner code:

In [48]:

from torch.utils.data import Dataset, DataLoader

class MyDataFromDF(Dataset):
    def __init__(self, X, y):
        # features provided as Dataframes converted
        self.X = torch.tensor(    
            X.to_numpy(), 
            # dtype=torch.float32() # optionally include a datatype
            ) 
        
        # targets provided as Dataframes
        self.y = torch.tensor( 
            y.to_numpy(),
            # dtype=torch.float32() # optionally include a datatype
            )            

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]   # always return (features, target)


In [49]:
# Wrap custom dataset in Dataloader

dataset = MyDataFromDF(X, y)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# for batch in loader:
#     print(batch)

In [50]:
# How to use the loader in training:
# for batch_X, batch_y in loader:
    #pred = model(batch_X)
    #loss = criterion(pred, batch_y)
    #...


Datatypes need to be handled by torch (integer, float, Boolean) - compatible dtypes will be converted into a common format (check with `print(tensor.dtype)`). A datatype can be specified in (`dtype=torch.float32`) - see some examples above.

Strings, categorical values are incompatible and need to be encoded into numerical values.

## Build a NN for Regression
For convenience and better comparison, the Diabetes dataset as available in scikit-learn is used here again.

Train-test split on original data:

In [51]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Convert to tensors - you could also use any other method from above! Note that here there was no DataLoader used as wrapper, as the dataset is small.

In [52]:
X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
X_test = torch.tensor(X_test.to_numpy(), dtype=torch.float32)
y_test = torch.tensor(y_test.to_numpy(), dtype=torch.float32)

A fully connected NN is defined as model class. Note the separation of the initialisation and the forward pass class functions (essentially separating the learning part from the activation functions).

In [53]:
import torch.nn as nn

class DiabetesNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(DiabetesNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  # input layer with width = length of the feature set
        self.fc2 = nn.Linear(hidden_size, hidden_size)  # one hidden layer, try to add another one
        self.fc3 = nn.Linear(hidden_size, output_size)  # output layer
    
    # Good practice: activation functions specified in forward pass - separated from the learning parts
    def forward(self, x):            # forward pass
        x = torch.relu(self.fc1(x))  # ReLU activation function for input layer
        x = torch.relu(self.fc2(x))  # ReLU activation function for hidden layer
        # no activation function for the output layer (because it is a regresion model!)
        return x

Define hyperparameters:

In [62]:
# Hyperparameters
input_size = X_train.shape[1]
hidden_size = 200
output_size = 1
learning_rate = 0.01

# Number of training iterations
num_epochs = 5000

Introduce loss function (``criterion``) and gradient optimiser (``optimizer``):

In [63]:
import torch.optim as optim
model = DiabetesNN(input_size, hidden_size, output_size)

criterion = nn.MSELoss()  # loss function defined;

optimizer = optim.Adam(model.parameters(), learning_rate) # gradient descent method based on average and squares of gradient

In [64]:
for epoch in range(num_epochs):
    
    model.train()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad() # clear existing gradients
    loss.backward() # backpropagation of loss function to optimise gradient
    optimizer.step() # update parameters using SDG

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}')

/Users/hassanhaydar/DSA104/.venv/lib/python3.12/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([353, 1])) that is different to the input size (torch.Size([353, 200])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [10/100], Loss: 29245.4277
Epoch [20/100], Loss: 28178.6484
Epoch [30/100], Loss: 26383.3340
Epoch [40/100], Loss: 23846.8965
Epoch [50/100], Loss: 20768.1602
Epoch [60/100], Loss: 17550.4629
Epoch [70/100], Loss: 14692.0967
Epoch [80/100], Loss: 12652.3633
Epoch [90/100], Loss: 11553.2441
Epoch [100/100], Loss: 11122.4873
Epoch [110/100], Loss: 10958.3916
Epoch [120/100], Loss: 10842.0947
Epoch [130/100], Loss: 10741.2412
Epoch [140/100], Loss: 10665.4385
Epoch [150/100], Loss: 10609.1787
Epoch [160/100], Loss: 10564.2930
Epoch [170/100], Loss: 10527.1201
Epoch [180/100], Loss: 10496.0713
Epoch [190/100], Loss: 10469.8984
Epoch [200/100], Loss: 10447.6299
Epoch [210/100], Loss: 10428.5752
Epoch [220/100], Loss: 10412.2031
Epoch [230/100], Loss: 10398.0723
Epoch [240/100], Loss: 10385.8115
Epoch [250/100], Loss: 10375.1133
Epoch [260/100], Loss: 10365.7217
Epoch [270/100], Loss: 10357.4229
Epoch [280/100], Loss: 10350.0439
Epoch [290/100], Loss: 10343.4414
Epoch [300/100], Loss: 

Evaluate the model on the test set to termine the generalisation for the regression model.

In [65]:
# Evaluate the model
mse_loss = nn.MSELoss()

model.eval()

with torch.no_grad():
    y_pred = model(X_test)
    mse = mse_loss(y_pred, y_test).item()
    print("MSE:", mse)


MSE: 2974.92822265625


/Users/hassanhaydar/DSA104/.venv/lib/python3.12/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([89, 1])) that is different to the input size (torch.Size([89, 200])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
